**View all teams**

In [1]:
from pyspark.sql import functions as F

df = spark.sql("select name, tla, crest from gold_teams order by name")
rows = df.collect()

cards = ""
for r in rows:
    cards += f"""
    <div style="border:1px solid #ddd;border-radius:10px;padding:12px 8px;display:flex;flex-direction:column;align-items:center;gap:6px;background:#fff;">
        <img src="{r['crest']}" width="52" height="52" style="object-fit:contain;" onerror="this.style.display='none'"/>
        <span style="font-size:11px;font-weight:600;text-align:center;color:#111;">{r['name']}</span>
        <span style="font-size:11px;color:#888;">{r['tla']}</span>
    </div>"""

html = f"""
<input id="s" placeholder="Buscar..." oninput="f()" style="margin-bottom:16px;padding:8px 12px;border:1px solid #ddd;border-radius:8px;width:300px;font-size:14px;"/>
<div id="grid" style="display:grid;grid-template-columns:repeat(auto-fill,minmax(110px,1fr));gap:12px;">{cards}</div>
<script>
function f(){{
  const q=document.getElementById('s').value.toLowerCase();
  document.querySelectorAll('#grid > div').forEach((el,i)=>{{
    el.style.display=el.innerText.toLowerCase().includes(q)?'':'none';
  }});
}}
</script>"""

displayHTML(html)

StatementMeta(, 55b3292a-cd65-4ac8-9691-c4b0481700e7, 3, Finished, Available, Finished, False)

**View all matches**

In [2]:
from pyspark.sql import functions as F
import json

rows = spark.sql("select * from gold_matches order by match_date, match_time_utc").collect()

matches = []
for r in rows:
    matches.append({
        "id": r["id"],
        "date": str(r["match_date"]),
        "time": r["match_time_utc"],
        "homeTeam": r["homeTeam_name"],
        "homeTla": r["homeTeam_tla"],
        "homeCrest": r["homeTeam_crest"],
        "awayTeam": r["awayTeam_name"],
        "awayTla": r["awayTeam_tla"],
        "awayCrest": r["awayTeam_crest"],
    })

matches_json = json.dumps(matches)

html = f"""
<style>
  * {{ box-sizing: border-box; margin: 0; padding: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; }}
  body {{ background: #f5f5f5; padding: 20px; }}
  .toolbar {{ display: flex; gap: 10px; margin-bottom: 20px; flex-wrap: wrap; align-items: center; }}
  .toolbar input {{ flex: 1; min-width: 180px; padding: 8px 14px; border: 1px solid #ddd; border-radius: 8px; font-size: 14px; background: #fff; }}
  .toolbar select {{ padding: 8px 12px; border: 1px solid #ddd; border-radius: 8px; font-size: 14px; background: #fff; cursor: pointer; }}
  .count {{ font-size: 13px; color: #888; margin-left: auto; white-space: nowrap; }}
  .date-group {{ margin-bottom: 24px; }}
  .date-label {{ font-size: 12px; font-weight: 600; color: #888; letter-spacing: .08em; text-transform: uppercase; margin-bottom: 10px; padding-left: 4px; }}
  .cards {{ display: grid; grid-template-columns: repeat(auto-fill, minmax(280px, 1fr)); gap: 10px; }}
  .card {{ background: #fff; border-radius: 12px; border: 1px solid #e8e8e8; padding: 14px 16px; display: flex; flex-direction: column; gap: 10px; }}
  .time-badge {{ font-size: 11px; font-weight: 600; color: #666; background: #f0f0f0; border-radius: 6px; padding: 3px 8px; align-self: flex-start; }}
  .matchup {{ display: flex; align-items: center; gap: 8px; }}
  .team {{ display: flex; flex-direction: column; align-items: center; gap: 5px; flex: 1; }}
  .team img {{ width: 44px; height: 44px; object-fit: contain; }}
  .team-name {{ font-size: 12px; font-weight: 600; color: #111; text-align: center; line-height: 1.3; }}
  .team-tla {{ font-size: 11px; color: #999; }}
  .vs {{ font-size: 13px; font-weight: 700; color: #bbb; flex-shrink: 0; }}
  .empty {{ text-align: center; color: #aaa; font-size: 14px; padding: 40px 0; }}
</style>

<div class="toolbar">
  <input type="text" id="search" placeholder="Buscar seleção..." oninput="render()" />
  <select id="dateFilter" onchange="render()">
    <option value="">Todos os dias</option>
  </select>
  <span class="count" id="count"></span>
</div>
<div id="container"></div>

<script>
const matches = {matches_json};

const dateFilter = document.getElementById('dateFilter');
const uniqueDates = [...new Set(matches.map(m => m.date))].sort();
uniqueDates.forEach(d => {{
  const opt = document.createElement('option');
  opt.value = d;
  const [y, mo, day] = d.split('-');
  opt.textContent = new Date(d + 'T12:00:00').toLocaleDateString('en-US', {{weekday:'short', day:'2-digit', month:'short'}});
  dateFilter.appendChild(opt);
}});

function fmtDate(d) {{
  return new Date(d + 'T12:00:00').toLocaleDateString('en-US', {{weekday:'long', day:'2-digit', month:'long'}});
}}

function render() {{
  const q = document.getElementById('search').value.toLowerCase();
  const df = dateFilter.value;
  let list = matches.filter(m => {{
    const matchQ = !q || m.homeTeam.toLowerCase().includes(q) || m.awayTeam.toLowerCase().includes(q) || m.homeTla.toLowerCase().includes(q) || m.awayTla.toLowerCase().includes(q);
    const matchD = !df || m.date === df;
    return matchQ && matchD;
  }});

  document.getElementById('count').textContent = list.length + ' jogo' + (list.length !== 1 ? 's' : '');

  const grouped = {{}};
  list.forEach(m => {{ if (!grouped[m.date]) grouped[m.date] = []; grouped[m.date].push(m); }});

  const container = document.getElementById('container');
  if (!list.length) {{ container.innerHTML = '<div class="empty">Nenhum jogo encontrado.</div>'; return; }}

  container.innerHTML = Object.keys(grouped).sort().map(date => `
    <div class="date-group">
      <div class="date-label">${{fmtDate(date)}}</div>
      <div class="cards">
        ${{grouped[date].map(m => `
          <div class="card">
            <span class="time-badge">⏱ ${{m.time}} UTC</span>
            <div class="matchup">
              <div class="team">
                <img src="${{m.homeCrest}}" alt="${{m.homeTeam}}" onerror="this.style.visibility='hidden'"/>
                <span class="team-name">${{m.homeTeam}}</span>
                <span class="team-tla">${{m.homeTla}}</span>
              </div>
              <span class="vs">VS</span>
              <div class="team">
                <img src="${{m.awayCrest}}" alt="${{m.awayTeam}}" onerror="this.style.visibility='hidden'"/>
                <span class="team-name">${{m.awayTeam}}</span>
                <span class="team-tla">${{m.awayTla}}</span>
              </div>
            </div>
          </div>`).join('')}}
      </div>
    </div>`).join('');
}}

render();
</script>
"""

displayHTML(html)


StatementMeta(, 55b3292a-cd65-4ac8-9691-c4b0481700e7, 4, Finished, Available, Finished, False)

**Matches by team**

In [3]:
from pyspark.sql import functions as F
import json

rows = spark.sql("select * from gold_matches order by match_date, match_time_utc").collect()

matches = []
for r in rows:
    matches.append({
        "id": r["id"],
        "date": str(r["match_date"]),
        "time": r["match_time_utc"],
        "homeTeam": r["homeTeam_name"],
        "homeTla": r["homeTeam_tla"],
        "homeCrest": r["homeTeam_crest"],
        "awayTeam": r["awayTeam_name"],
        "awayTla": r["awayTeam_tla"],
        "awayCrest": r["awayTeam_crest"],
    })

matches_json = json.dumps(matches)

html = f"""
<style>
  * {{ box-sizing: border-box; margin: 0; padding: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; }}
  body {{ background: #f5f5f5; padding: 20px; }}
  .toolbar {{ display: flex; gap: 12px; align-items: center; margin-bottom: 24px; flex-wrap: wrap; }}
  .team-select {{ display: flex; align-items: center; gap: 10px; background: #fff; border: 1px solid #e0e0e0; border-radius: 10px; padding: 8px 14px; cursor: pointer; min-width: 260px; }}
  .team-select img {{ width: 32px; height: 32px; object-fit: contain; }}
  .team-select select {{ border: none; outline: none; font-size: 15px; font-weight: 500; color: #111; background: transparent; cursor: pointer; flex: 1; }}
  .count {{ font-size: 13px; color: #888; }}
  .cards {{ display: grid; grid-template-columns: repeat(auto-fill, minmax(280px, 1fr)); gap: 10px; }}
  .card {{ background: #fff; border-radius: 12px; border: 1px solid #e8e8e8; padding: 14px 16px; display: flex; flex-direction: column; gap: 10px; }}
  .card.highlight {{ border-color: #4CAF50; border-width: 2px; }}
  .date-label {{ font-size: 12px; font-weight: 600; color: #888; letter-spacing: .08em; text-transform: uppercase; margin: 16px 0 8px 4px; }}
  .time-badge {{ font-size: 11px; font-weight: 600; color: #666; background: #f0f0f0; border-radius: 6px; padding: 3px 8px; align-self: flex-start; }}
  .matchup {{ display: flex; align-items: center; gap: 8px; }}
  .team {{ display: flex; flex-direction: column; align-items: center; gap: 5px; flex: 1; }}
  .team img {{ width: 44px; height: 44px; object-fit: contain; }}
  .team-name {{ font-size: 12px; font-weight: 600; color: #111; text-align: center; line-height: 1.3; }}
  .team-tla {{ font-size: 11px; color: #999; }}
  .team-name.selected {{ color: #2e7d32; }}
  .vs {{ font-size: 13px; font-weight: 700; color: #bbb; flex-shrink: 0; }}
  .empty {{ text-align: center; color: #aaa; font-size: 14px; padding: 40px 0; }}
</style>

<div class="toolbar">
  <div class="team-select">
    <img id="selectedCrest" src="" style="display:none" />
    <select id="teamSelect" onchange="render()">
      <option value="">Todas as seleções</option>
    </select>
  </div>
  <span class="count" id="count"></span>
</div>
<div id="container"></div>

<script>
const matches = {matches_json};

const teams = {{}};
matches.forEach(m => {{
  teams[m.homeTeam] = m.homeCrest;
  teams[m.awayTeam] = m.awayCrest;
}});

const sel = document.getElementById('teamSelect');
Object.keys(teams).sort().forEach(name => {{
  const opt = document.createElement('option');
  opt.value = name;
  opt.textContent = name;
  sel.appendChild(opt);
}});

function fmtDate(d) {{
  return new Date(d + 'T12:00:00').toLocaleDateString('en-US', {{weekday:'long', day:'2-digit', month:'long'}});
}}

function render() {{
  const team = sel.value;
  const crestImg = document.getElementById('selectedCrest');
  if (team) {{
    crestImg.src = teams[team];
    crestImg.style.display = 'block';
  }} else {{
    crestImg.style.display = 'none';
  }}

  const list = matches.filter(m => !team || m.homeTeam === team || m.awayTeam === team);
  document.getElementById('count').textContent = list.length + ' jogo' + (list.length !== 1 ? 's' : '');

  const grouped = {{}};
  list.forEach(m => {{ if (!grouped[m.date]) grouped[m.date] = []; grouped[m.date].push(m); }});

  const container = document.getElementById('container');
  if (!list.length) {{ container.innerHTML = '<div class="empty">Nenhum jogo encontrado.</div>'; return; }}

  container.innerHTML = Object.keys(grouped).sort().map(date => `
    <div class="date-label">${{fmtDate(date)}}</div>
    <div class="cards">
      ${{grouped[date].map(m => {{
        const isHome = m.homeTeam === team;
        const isAway = m.awayTeam === team;
        return `
          <div class="card ${{team ? 'highlight' : ''}}">
            <span class="time-badge">⏱ ${{m.time}} UTC</span>
            <div class="matchup">
              <div class="team">
                <img src="${{m.homeCrest}}" onerror="this.style.visibility='hidden'"/>
                <span class="team-name ${{isHome ? 'selected' : ''}}">${{m.homeTeam}}</span>
                <span class="team-tla">${{m.homeTla}}</span>
              </div>
              <span class="vs">VS</span>
              <div class="team">
                <img src="${{m.awayCrest}}" onerror="this.style.visibility='hidden'"/>
                <span class="team-name ${{isAway ? 'selected' : ''}}">${{m.awayTeam}}</span>
                <span class="team-tla">${{m.awayTla}}</span>
              </div>
            </div>
          </div>`;
      }}).join('')}}
    </div>`).join('');
}}

render();
</script>
"""

displayHTML(html)

StatementMeta(, 55b3292a-cd65-4ac8-9691-c4b0481700e7, 5, Finished, Available, Finished, False)

**Players by country**

In [7]:
from pyspark.sql import functions as F
import json

rows = spark.sql("select * from gold_players order by team_name, position, name").collect()

# Busca os crests da gold_teams
crests = {r["name"]: r["crest"] for r in spark.sql("select name, crest from gold_teams").collect()}

players = []
for r in rows:
    players.append({
        "id": r["id_player"],
        "name": r["name"],
        "dob": r["dateOfBirth"] or "",
        "age": r["age"] if r["age"] is not None else "",
        "nationality": r["nationality"] or "",
        "position": r["position"] or "—",
        "team": r["team_name"],
        "tla": r["team_tla"],
    })

teams = {}
for p in players:
    if p["team"] not in teams:
        teams[p["team"]] = {"tla": p["tla"], "crest": crests.get(p["team"], "")}

players_json = json.dumps(players)
teams_json = json.dumps(teams)

position_order = [
    "Goalkeeper","Defence","Left-Back","Right-Back","Centre-Back",
    "Defensive Midfield","Central Midfield","Attacking Midfield",
    "Midfield","Left Winger","Right Winger","Right Midfield","Left Midfield",
    "Centre-Forward","Offence","null","—"
]
position_order_json = json.dumps(position_order)

html = f"""
<style>
  * {{ box-sizing: border-box; margin: 0; padding: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; }}
  body {{ background: #f5f5f5; padding: 20px; }}
  .toolbar {{ display: flex; gap: 12px; align-items: center; margin-bottom: 20px; flex-wrap: wrap; }}
  .team-select {{ display: flex; align-items: center; gap: 10px; background: #fff; border: 1px solid #e0e0e0; border-radius: 10px; padding: 8px 14px; min-width: 280px; }}
  .team-select img {{ width: 32px; height: 32px; object-fit: contain; }}
  .team-select select {{ border: none; outline: none; font-size: 15px; font-weight: 500; color: #111; background: transparent; cursor: pointer; flex: 1; }}
  .search-box {{ flex: 1; min-width: 160px; padding: 8px 14px; border: 1px solid #e0e0e0; border-radius: 10px; font-size: 14px; background: #fff; outline: none; }}
  .count {{ font-size: 13px; color: #888; white-space: nowrap; }}
  .group {{ margin-bottom: 20px; }}
  .group-label {{ font-size: 11px; font-weight: 700; color: #999; letter-spacing: .1em; text-transform: uppercase; margin-bottom: 8px; padding-left: 2px; }}
  table {{ width: 100%; border-collapse: collapse; background: #fff; border-radius: 12px; overflow: hidden; border: 1px solid #eee; }}
  th {{ font-size: 11px; font-weight: 600; color: #aaa; text-transform: uppercase; letter-spacing: .05em; padding: 8px 12px; text-align: left; border-bottom: 1px solid #f0f0f0; background: #fafafa; }}
  td {{ font-size: 13px; color: #222; padding: 9px 12px; border-bottom: 1px solid #f5f5f5; vertical-align: middle; }}
  tr:last-child td {{ border-bottom: none; }}
  tr:hover td {{ background: #fafafa; }}
  .badge {{ display: inline-block; font-size: 10px; font-weight: 600; padding: 2px 7px; border-radius: 20px; background: #f0f0f0; color: #555; }}
  .age {{ color: #999; font-size: 12px; }}
  .empty {{ text-align: center; color: #aaa; font-size: 14px; padding: 40px 0; }}
  .team-header {{ display: flex; align-items: center; gap: 10px; margin-bottom: 16px; }}
  .team-header img {{ width: 40px; height: 40px; object-fit: contain; }}
  .team-header h2 {{ font-size: 18px; font-weight: 600; color: #111; }}
  .team-header span {{ font-size: 13px; color: #999; }}
</style>

<div class="toolbar">
  <div class="team-select">
    <img id="selectedCrest" src="" style="display:none" />
    <select id="teamSelect" onchange="render()">
      <option value="">All teams</option>
    </select>
  </div>
  <input class="search-box" type="text" id="search" placeholder="Search Player..." oninput="render()" />
  <span class="count" id="count"></span>
</div>
<div id="container"></div>

<script>
const players = {players_json};
const teams = {teams_json};
const posOrder = {position_order_json};

const sel = document.getElementById('teamSelect');
Object.keys(teams).sort().forEach(name => {{
  const opt = document.createElement('option');
  opt.value = name;
  opt.textContent = name + ' (' + teams[name].tla + ')';
  sel.appendChild(opt);
}});

function posSort(a, b) {{
  const ia = posOrder.indexOf(a) === -1 ? 999 : posOrder.indexOf(a);
  const ib = posOrder.indexOf(b) === -1 ? 999 : posOrder.indexOf(b);
  return ia - ib;
}}

function render() {{
  const team = sel.value;
  const q = document.getElementById('search').value.toLowerCase();
  const crestImg = document.getElementById('selectedCrest');

  if (team) {{
    crestImg.src = teams[team].crest;
    crestImg.style.display = 'block';
  }} else {{
    crestImg.style.display = 'none';
  }}

  let list = players.filter(p => {{
    const matchT = !team || p.team === team;
    const matchQ = !q || p.name.toLowerCase().includes(q) || p.nationality.toLowerCase().includes(q) || p.position.toLowerCase().includes(q);
    return matchT && matchQ;
  }});

  document.getElementById('count').textContent = list.length + ' player' + (list.length !== 1 ? 'es' : '');

  const container = document.getElementById('container');
  if (!list.length) {{ container.innerHTML = '<div class="empty">No player found.</div>'; return; }}

  // Agrupa por Position se filtrou por time, senão por time
  if (team) {{
    const grouped = {{}};
    list.forEach(p => {{
      const pos = p.position;
      if (!grouped[pos]) grouped[pos] = [];
      grouped[pos].push(p);
    }});
    const sortedPos = Object.keys(grouped).sort(posSort);
    container.innerHTML = `
      <div class="team-header">
        <img src="${{teams[team].crest}}" />
        <h2>${{team}}</h2>
        <span>${{teams[team].tla}} · ${{list.length}} jogadores</span>
      </div>
      ${{sortedPos.map(pos => `
        <div class="group">
          <div class="group-label">${{pos}}</div>
          <table>
            <thead><tr><th>#</th><th>Name</th><th>Nasc.</th><th>Age</th><th>Birth place</th></tr></thead>
            <tbody>
              ${{grouped[pos].map(p => `
                <tr>
                  <td style="color:#ccc;width:30px">${{p.id}}</td>
                  <td style="font-weight:500">${{p.name}}</td>
                  <td style="color:#999">${{p.dob}}</td>
                  <td><span class="age">${{p.age}}</span></td>
                  <td><span class="badge">${{p.nationality}}</span></td>
                </tr>`).join('')}}
            </tbody>
          </table>
        </div>`).join('')}}`;
  }} else {{
    const grouped = {{}};
    list.forEach(p => {{
      if (!grouped[p.team]) grouped[p.team] = [];
      grouped[p.team].push(p);
    }});
    container.innerHTML = Object.keys(grouped).sort().map(t => `
      <div class="group">
        <div class="group-label" style="display:flex;align-items:center;gap:6px;">
          <img src="${{teams[t].crest}}" style="width:16px;height:16px;object-fit:contain;"/>
          ${{t}} <span style="font-weight:400">(${{grouped[t].length}})</span>
        </div>
        <table>
          <thead><tr><th>Name</th><th>Position</th><th>Age</th><th>Birth place</th></tr></thead>
          <tbody>
            ${{grouped[t].map(p => `
              <tr>
                <td style="font-weight:500">${{p.name}}</td>
                <td><span class="badge">${{p.position}}</span></td>
                <td><span class="age">${{p.age}}</span></td>
                <td style="color:#999">${{p.nationality}}</td>
              </tr>`).join('')}}
          </tbody>
        </table>
      </div>`).join('');
  }}
}}

render();
</script>
"""

displayHTML(html)


StatementMeta(, 55b3292a-cd65-4ac8-9691-c4b0481700e7, 9, Finished, Available, Finished, False)

**Average age by team**

In [5]:
from pyspark.sql import functions as F

df_avg_age = (spark.sql("select * from gold_players")
    .groupBy("team_name", "team_tla")
    .agg(F.round(F.avg("age"), 1).alias("avg_age"))
    .orderBy("avg_age", ascending=False)
)

display(df_avg_age)

StatementMeta(, 55b3292a-cd65-4ac8-9691-c4b0481700e7, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5ad47c4e-e097-4773-8030-7b42e7503703)